In [1]:
import pandas as pd
import numpy as np
data=pd.read_csv('t20_features_cricsheet.csv.gz')

In [2]:
data.columns

Index(['batted', 'bat_pos', 'runs', 'balls_faced', 'fours', 'sixes',
       'dot_faced', 'is_out', 'bat_pp_balls', 'bat_death_balls',
       ...
       'fld_catches_rate', 'fld_stumpings_rate', 'fld_run_outs_rate',
       'matches_prev', 'days_since', 'venue_avg_total', 'team_bat_form',
       'team_conceded_form', 'opp_conceded_form', 'opp_bat_form'],
      dtype='object', length=116)

In [1]:
import os, json, glob, pandas as pd, numpy as np
from tqdm import tqdm


T20_DIR = 't20_I_json'
IPL_DIR = 'ipl_json'

t20_files = sorted(glob.glob(os.path.join(T20_DIR, '*.json')))
ipl_files = sorted(glob.glob(os.path.join(IPL_DIR, '*.json')))
print('t20 files:', len(t20_files), '| ipl files:', len(ipl_files))   

WK = {'bowled','caught','caught and bowled','lbw','stumped','hit wicket'}
NOT_OUT = {'retired hurt','retired not out'}

def parse(fp, src):
    try:
        d = json.load(open(fp, encoding='utf-8'))
    except Exception:
        return None
    i = d['info']
    if i.get('gender') != 'male' or 'players' not in i: return None
    reg = i.get('registry', {}).get('people', {})
    mid = os.path.basename(fp)[:-5]
    date = i['dates'][0]
    teams = i['teams']
    st = {}
    def S(team, p):
        k = (team, p)
        if k not in st:
            st[k] = dict(batted=0, bat_pos=np.nan, runs=0, balls_faced=0, fours=0, sixes=0, dot_faced=0, is_out=0,
                         bat_pp_balls=0, bat_death_balls=0, balls_bowled=0, runs_conceded=0, wickets=0, dot_bowled=0,
                         wides=0, noballs=0, bowl_pp_balls=0, bowl_death_balls=0, catches=0, stumpings=0, run_outs=0)
        return st[k]
    team_tot = {}; first_bat = None
    for inn in d['innings']:
        if inn.get('super_over'): continue
        bt = inn['team']; bo = [t for t in teams if t != bt][0]
        if first_bat is None: first_bat = bt
        order = []; tot = 0
        for ov in inn['overs']:
            oi = ov['over']
            for dl in ov['deliveries']:
                b, nb, bw = dl['batter'], dl['non_striker'], dl['bowler']
                for x in (b, nb):
                    if x not in order:
                        order.append(x); s = S(bt, x); s['batted'] = 1; s['bat_pos'] = len(order)
                r = dl['runs']; ex = dl.get('extras', {})
                tot += r['total']
                wide = 'wides' in ex; nobl = 'noballs' in ex
                sb = S(bt, b)
                if not wide:
                    sb['balls_faced'] += 1
                    if r['batter'] == 0: sb['dot_faced'] += 1
                    if oi < 6: sb['bat_pp_balls'] += 1
                    if oi >= 15: sb['bat_death_balls'] += 1
                sb['runs'] += r['batter']
                if r['batter'] == 4: sb['fours'] += 1
                if r['batter'] == 6: sb['sixes'] += 1
                sw = S(bo, bw)
                conc = r['total'] - ex.get('byes', 0) - ex.get('legbyes', 0)
                sw['runs_conceded'] += conc
                if wide: sw['wides'] += 1
                if nobl: sw['noballs'] += 1
                if not wide and not nobl:
                    sw['balls_bowled'] += 1
                    if conc == 0: sw['dot_bowled'] += 1
                    if oi < 6: sw['bowl_pp_balls'] += 1
                    if oi >= 15: sw['bowl_death_balls'] += 1
                for w in dl.get('wickets', []):
                    k = w['kind']
                    if k not in NOT_OUT:
                        S(bt, w['player_out'])['is_out'] = 1
                    if k in WK: sw['wickets'] += 1
                    for f in w.get('fielders', []):
                        if f.get('substitute'): continue
                        fn = f.get('name')
                        if fn is None: continue
                        sf = S(bo, fn)
                        if k in ('caught', 'caught and bowled'): sf['catches'] += 1
                        elif k == 'stumped': sf['stumpings'] += 1
                        elif k == 'run out': sf['run_outs'] += 1
        team_tot[bt] = tot
    rows = []; win = i['outcome'].get('winner')
    toss = i.get('toss', {})
    for t in teams:
        opp = [x for x in teams if x != t][0]
        inno = 1 if t == first_bat else 2
        for p in i['players'].get(t, []):
            s = dict(S(t, p))
            s.update(match_id=mid, date=date, venue=i.get('venue'), city=i.get('city'),
                     event=(i.get('event') or {}).get('name'), season=str(i.get('season')), src=src,
                     team=t, opp=opp, innings=inno, toss_won=int(toss.get('winner') == t),
                     toss_decision=toss.get('decision'), player_id=reg.get(p, p), player=p,
                     team_total=team_tot.get(t, np.nan), opp_total=team_tot.get(opp, np.nan),
                     team_won=(np.nan if win is None else int(win == t)))
            rows.append(s)
    return rows

rows = []
for fp in tqdm(t20_files, desc='T20I'):
    r = parse(fp, 'intl')
    if r: rows.extend(r)
for fp in tqdm(ipl_files, desc='IPL'):
    r = parse(fp, 'ipl')
    if r: rows.extend(r)

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
df['overs_bowled'] = (df.balls_bowled // 6) + (df.balls_bowled % 6) / 10
df.to_pickle('raw_players.pkl')
print(df.shape, df.match_id.nunique(), df.player_id.nunique())
print(df.src.value_counts())

t20 files: 3592 | ipl files: 1243


IPL: 100%|█████████████████████████████████████████████████████████████████████████| 1243/1243 [00:18<00:00, 66.63it/s]


(106942, 39) 4835 5134
src
intl    79033
ipl     27909
Name: count, dtype: int64


In [2]:
import pandas as pd, numpy as np

df = pd.read_pickle('raw_players.pkl')

# ---- phantom "as-of" rows: har player ki ek nakli row date 2100 ki ----
# isse hum kisi bhi date par "is player ke features us din tak ke" nikaal sakte hain
ph = df.sort_values('date').groupby('player_id').tail(1).copy()
for c in ['batted','runs','balls_faced','fours','sixes','dot_faced','is_out','bat_pp_balls','bat_death_balls',
          'balls_bowled','runs_conceded','wickets','dot_bowled','wides','noballs','bowl_pp_balls',
          'bowl_death_balls','catches','stumpings','run_outs','overs_bowled']:
    ph[c] = 0
ph['bat_pos'] = np.nan
ph['date'] = pd.Timestamp('2100-01-01'); ph['match_id'] = 'PHANTOM'
ph['team'] = np.nan; ph['opp'] = np.nan; ph['team_total'] = np.nan; ph['opp_total'] = np.nan
ph['venue'] = np.nan; ph['src'] = 'intl'; ph['toss_won'] = 0; ph['toss_decision'] = 'field'; ph['innings'] = 1
df = pd.concat([df, ph], ignore_index=True)

df['is_intl'] = (df.src == 'intl').astype(int)
df['bat_first_dec'] = ((df.toss_decision == 'bat') == (df.toss_won == 1)).astype(int)
df = df.sort_values(['player_id','date','match_id']).reset_index(drop=True)

TRAIN_END = pd.Timestamp('2024-01-01')
tr = df[df.date < TRAIN_END]
bat_cols  = ['runs','balls_faced','fours','sixes','dot_faced','is_out','bat_pos','bat_pp_balls','bat_death_balls']
bowl_cols = ['balls_bowled','runs_conceded','wickets','dot_bowled','wides','noballs','bowl_pp_balls','bowl_death_balls']
fld_cols  = ['catches','stumpings','run_outs']
prior_bat  = tr[tr.batted == 1][bat_cols].mean()
prior_bowl = tr[tr.balls_bowled > 0][bowl_cols].mean()
K = 5   # shrinkage: prior ka weight = 5 matches
g = df.groupby('player_id')

def roll(mask, cols, W, tag, prior, n_name):
    """pichle W matches ka mean (sirf wahi matches jahan mask==1), pichle rows se hi (leak-free), prior ki taraf shrunk"""
    out = {}
    m = mask.astype(float)
    cm = m.groupby(df.player_id).cumsum()
    for c in cols:
        x = df[c].fillna(0) * m
        cx = x.groupby(df.player_id).cumsum()
        if W is None:
            S = cx.groupby(df.player_id).shift(1)
            N = cm.groupby(df.player_id).shift(1)
        else:
            S = cx.groupby(df.player_id).shift(1) - cx.groupby(df.player_id).shift(W+1)
            N = cm.groupby(df.player_id).shift(1) - cm.groupby(df.player_id).shift(W+1)
            S = S.where(cx.groupby(df.player_id).shift(W+1).notna(), cx.groupby(df.player_id).shift(1))
            N = N.where(cm.groupby(df.player_id).shift(W+1).notna(), cm.groupby(df.player_id).shift(1))
        S = S.fillna(0); N = N.fillna(0)
        out[f'{tag}_{c}_{W or "all"}'] = (S + K*prior[c]) / (N + K)
        out[f'{n_name}_{W or "all"}'] = N
    return pd.DataFrame(out)

batted = df.batted
bowled = (df.balls_bowled > 0)
parts = []
for W in (5, 15, None):
    parts.append(roll(batted, bat_cols, W, 'bat', prior_bat, 'n_bat'))
    parts.append(roll(bowled, bowl_cols, W, 'bowl', prior_bowl, 'n_bowl'))
F = pd.concat(parts, axis=1)

for W in ('5','15','all'):
    F[f'bat_sr_{W}']    = F[f'bat_runs_{W}'] / F[f'bat_balls_faced_{W}'].clip(lower=1) * 100
    F[f'bowl_econ_{W}'] = F[f'bowl_runs_conceded_{W}'] / (F[f'bowl_balls_bowled_{W}'].clip(lower=1) / 6)

one = pd.Series(1.0, index=df.index)
def share(mask, W):
    x = mask.astype(float)
    cx = x.groupby(df.player_id).cumsum(); c1 = one.groupby(df.player_id).cumsum()
    S = cx.groupby(df.player_id).shift(1) - cx.groupby(df.player_id).shift(W+1).fillna(0)
    N = c1.groupby(df.player_id).shift(1) - c1.groupby(df.player_id).shift(W+1).fillna(0)
    return (S.fillna(0) / N.replace(0, np.nan)).fillna(0)
F['p_bat_15']  = share(batted, 15)
F['p_bowl_15'] = share(bowled, 15)

for c in fld_cols:
    x = df[c]; cx = x.groupby(df.player_id).cumsum(); c1 = one.groupby(df.player_id).cumsum()
    F[f'fld_{c}_rate'] = cx.groupby(df.player_id).shift(1).fillna(0) / (c1.groupby(df.player_id).shift(1).fillna(0) + K)

F['matches_prev'] = g.cumcount()
F['days_since']   = g['date'].diff().dt.days.fillna(365).clip(upper=365)

# ---- venue aur team-level context (sab past-only) ----
inn = df.drop_duplicates(['match_id','team'])[['match_id','date','venue','team','opp','team_total']] \
        .dropna(subset=['team_total']).sort_values(['date','match_id'])
gl = inn[inn.date < TRAIN_END].team_total.mean()

def past_mean(keys, val, K2=5, prior=gl):
    grp = inn.groupby(keys)[val]
    S = grp.cumsum() - inn[val]; N = grp.cumcount()
    return (S + K2*prior) / (N + K2)
inn['venue_avg_total'] = past_mean('venue', 'team_total')

def team_roll(col_by, col_val, name, W=20):
    r = inn.groupby(col_by)[col_val].transform(lambda s: s.shift(1).rolling(W, min_periods=3).mean())
    inn[name] = r.fillna(gl)

inn['runs_against'] = inn.merge(
    inn[['match_id','team','team_total']].rename(columns={'team':'opp','team_total':'opp_t'}),
    on=['match_id','opp'], how='left')['opp_t'].values
team_roll('team', 'team_total', 'team_bat_form')
team_roll('team', 'runs_against', 'team_conceded_form')

opp_form = inn[['match_id','team','team_conceded_form','team_bat_form']].rename(
    columns={'team':'opp','team_conceded_form':'opp_conceded_form','team_bat_form':'opp_bat_form'})
ctx = inn[['match_id','team','venue_avg_total','team_bat_form','team_conceded_form']]

df2 = pd.concat([df, F], axis=1)
df2 = df2.merge(ctx, on=['match_id','team'], how='left').merge(opp_form, on=['match_id','opp'], how='left')
for c in ['venue_avg_total','team_bat_form','team_conceded_form','opp_conceded_form','opp_bat_form']:
    df2[c] = df2[c].fillna(gl)

# selection ke liye do extra columns
df2['prev_date']  = df2.groupby('player_id')['date'].shift(1)
df2['stump_prev'] = df2.groupby('player_id')['stumpings'].cumsum() - df2['stumpings']

df2.to_pickle('feat_all.pkl')
print(df2.shape, (df2.match_id == 'PHANTOM').sum(), 'phantom rows')

(112076, 118) 5134 phantom rows


In [3]:
import pandas as pd, numpy as np, warnings, itertools
warnings.filterwarnings('ignore')
import xgboost as xgb
from sklearn.metrics import mean_absolute_error as MAE
from pymoo.core.problem import Problem
from pymoo.core.repair import Repair
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.ux import UniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize import minimize

rng = np.random.default_rng(0)
f = pd.read_pickle('feat_all.pkl')
TEST = pd.Timestamp('2025-01-01')
real = f[f.match_id != 'PHANTOM']

# ---- features (toss aur innings nahi, kyunki selection toss se pehle hota hai) ----
common = ['is_intl','days_since','matches_prev','venue_avg_total','p_bat_15','p_bowl_15']
bat_f  = common + ['team_bat_form','opp_conceded_form'] + \
         [c for c in f.columns if c.startswith('bat_') and c.split('_')[-1] in ('5','15','all')] + \
         [c for c in f.columns if c.startswith('n_bat_')]
bowl_f = common + ['team_conceded_form','opp_bat_form'] + \
         [c for c in f.columns if c.startswith('bowl_') and c.split('_')[-1] in ('5','15','all')] + \
         [c for c in f.columns if c.startswith('n_bowl_')]
bat_f, bowl_f = list(dict.fromkeys(bat_f)), list(dict.fromkeys(bowl_f))

# ---- final scoring models: 2025 se pehle ke saare players par train ----
def fit(mask, feats, target, hp, nt):
    tr = real[mask & (real.date < TEST)]
    return [xgb.XGBRegressor(objective='count:poisson', n_estimators=nt, subsample=0.8, colsample_bytree=0.7,
                             reg_lambda=5, random_state=s, **hp).fit(tr[feats], tr[target]) for s in range(3)]
mb = fit(real.batted == 1, bat_f, 'runs', dict(max_depth=4, learning_rate=0.03, min_child_weight=30), 386)
mw = fit(real.balls_bowled > 0, bowl_f, 'wickets', dict(max_depth=2, learning_rate=0.03, min_child_weight=30), 717)
pred = lambda ms, X: np.mean([m.predict(X) for m in ms], 0)

te = real[real.date >= TEST]
tb, tw = te[te.batted == 1], te[te.balls_bowled > 0]
print('sanity | bat MAE base=%.3f model=%.3f | bowl MAE base=%.3f model=%.3f' % (
    MAE(tb.runs, tb.bat_runs_15), MAE(tb.runs, pred(mb, tb[bat_f])),
    MAE(tw.wickets, tw.bowl_wickets_15), MAE(tw.wickets, pred(mw, tw[bowl_f]))))

# ---- India T20I matches ----
ind = real[(real.team == 'India') & (real.src == 'intl')]
matches = ind.groupby('match_id').agg(date=('date','first'), event=('event','first')).sort_values('date')
xi = {m: set(g.player_id) for m, g in ind.groupby('match_id')}
names = dict(zip(real.player_id, real.player))

fs = f.sort_values(['player_id','date']).reset_index(drop=True)
pdates = {p: g.date.values for p, g in fs.groupby('player_id')}
first_pos = fs.groupby('player_id').cumcount() == 0
base_idx = dict(zip(fs.loc[first_pos, 'player_id'], fs.index[first_pos]))
def asof_row(p, date):
    j = np.searchsorted(pdates[p], np.datetime64(date), side='left')
    return base_idx[p] + j

# ---- exact Pareto front (pool chhota hai, to saare combinations enumerate) ----
def pareto_exact(bat, bowl, keeper, bowler, k=11):
    n = len(bat)
    C = np.array(list(itertools.combinations(range(n), k)), dtype=np.int16)
    ok = (keeper[C].sum(1) >= 1) & (bowler[C].sum(1) >= 5)
    C = C[ok]
    if len(C) == 0: return None
    b1, b2 = bat[C].sum(1), bowl[C].sum(1)
    o = np.lexsort((-b2, -b1)); best = -1; keep = []
    for i in o:
        if b2[i] > best + 1e-12: keep.append(i); best = b2[i]
    keep = np.array(keep)
    return C, C[keep], b1[keep], b2[keep], b1, b2

def knee(b1, b2):
    if len(b1) < 3: return int(np.argmax(b1/b1.max() + b2/max(b2.max(), 1e-9)))
    x = (b1 - b1.min()) / max(np.ptp(b1), 1e-9); y = (b2 - b2.min()) / max(np.ptp(b2), 1e-9)
    i0, i1 = np.argmax(x), np.argmax(y)
    a = np.array([x[i0], y[i0]]); b = np.array([x[i1], y[i1]]); d = b - a
    if np.linalg.norm(d) < 1e-9: return int(np.argmax(x + y))
    dist = np.abs((d[0]*(y - a[1]) - d[1]*(x - a[0])) / np.linalg.norm(d))
    return int(np.argmax(dist))

def hv(b1, b2):
    o = np.argsort(-b1); area = 0; prev = 0
    for i in o:
        if b2[i] > prev: area += b1[i] * (b2[i] - prev); prev = b2[i]
    return area

# ---- NSGA-II ----
class SubsetSampling(Sampling):
    def _do(self, problem, n_samples, **kw):
        X = np.zeros((n_samples, problem.n_var), dtype=bool)
        for r in range(n_samples): X[r, rng.choice(problem.n_var, 11, replace=False)] = True
        return X
class Fix11(Repair):
    def _do(self, problem, X, **kw):
        X = X.astype(bool).copy()
        for r in range(len(X)):
            s = X[r].sum()
            if s > 11: X[r, rng.choice(np.where(X[r])[0], s-11, replace=False)] = False
            elif s < 11: X[r, rng.choice(np.where(~X[r])[0], 11-s, replace=False)] = True
        return X
class TeamProb(Problem):
    def __init__(s, bat, bowl, keeper, bowler):
        super().__init__(n_var=len(bat), n_obj=2, n_ieq_constr=2, xl=0, xu=1, vtype=bool)
        s.bat, s.bowl, s.k, s.b = bat, bowl, keeper, bowler
    def _evaluate(s, X, out, *a, **kw):
        X = X.astype(float)
        out['F'] = np.column_stack([-(X @ s.bat), -(X @ s.bowl)])
        out['G'] = np.column_stack([1 - X @ s.k, 5 - X @ s.b])
def nsga2(bat, bowl, keeper, bowler, seed=1):
    alg = NSGA2(pop_size=80, sampling=SubsetSampling(), crossover=UniformCrossover(prob=0.9),
                mutation=BitflipMutation(prob=0.3, prob_var=1.0/len(bat)*2), repair=Fix11(), eliminate_duplicates=True)
    r = minimize(TeamProb(bat, bowl, keeper, bowler), alg, ('n_gen', 60), seed=seed, verbose=False)
    if r.X is None: return None
    X = np.atleast_2d(r.X).astype(bool); F = -np.atleast_2d(r.F)
    return X, F[:, 0], F[:, 1]

# ---- backtest: 2025+ ke har India T20I ke liye ----
rows = []; detail = []
for mid, mr in matches[matches.date >= TEST].iterrows():
    win = matches[(matches.event == mr.event) & ((matches.date - mr.date).abs() <= pd.Timedelta(days=45))]
    pool = sorted(set().union(*[xi[m] for m in win.index]))
    if len(pool) < 11 or len(pool) > 24:
        rows.append(dict(match_id=mid, skip=f'pool={len(pool)}')); continue
    ctxrow = ind[ind.match_id == mid].iloc[0]
    pos = [asof_row(p, mr.date) for p in pool]
    X = fs.loc[pos].copy().reset_index(drop=True)
    X['days_since'] = ((mr.date - X.prev_date).dt.days.fillna(365)).clip(upper=365)
    X['is_intl'] = 1
    for c in ['venue_avg_total','team_bat_form','team_conceded_form','opp_conceded_form','opp_bat_form']:
        X[c] = ctxrow[c]
    keeper = (X.stump_prev.values >= 2).astype(float)
    bowler = (X.p_bowl_15.values >= 0.35).astype(float)
    S = {}
    S['model'] = (X.p_bat_15.values * pred(mb, X[bat_f]), X.p_bowl_15.values * pred(mw, X[bowl_f]))
    S['form']  = (X.p_bat_15.values * X.bat_runs_15.values, X.p_bowl_15.values * X.bowl_wickets_15.values)
    actual = xi[mid]; idx = {p: i for i, p in enumerate(pool)}
    act_ix = np.array([idx[p] for p in actual if p in idx])
    ex = pareto_exact(*S['model'], keeper, bowler)
    if ex is None: rows.append(dict(match_id=mid, skip='infeasible')); continue
    C, FC, fb1, fb2, _, _ = ex
    ns = nsga2(*S['model'], keeper, bowler)
    if ns is None: rows.append(dict(match_id=mid, skip='nsga-none')); continue
    Xn, nb1, nb2 = ns
    pick_model = set(np.where(Xn[knee(nb1, nb2)])[0])
    pick_exact = set(FC[knee(fb1, fb2)])
    exf = pareto_exact(*S['form'], keeper, bowler)
    pick_form = set(exf[1][knee(exf[2], exf[3])])
    prev = matches[matches.date < mr.date].index[-1]; prevxi = xi[prev]
    last10 = matches[matches.date < mr.date].index[-10:]
    cnt = pd.Series([p for m in last10 for p in xi[m]]).value_counts()
    freq = set(cnt.index[:11])
    rnd = C[rng.choice(len(C), min(300, len(C)), replace=False)]
    act = set(act_ix); ov = lambda s: len(s & act)
    rand_ov = np.mean([len(set(r) & act) for r in rnd])
    a1, a2 = S['model'][0][act_ix].sum(), S['model'][1][act_ix].sum()
    dominated = bool(((fb1 >= a1-1e-9) & (fb2 >= a2-1e-9) & ((fb1 > a1+1e-9) | (fb2 > a2+1e-9))).any())
    rows.append(dict(match_id=mid, date=mr.date.date(), event=mr.event, pool=len(pool),
        front_exact=len(fb1), front_nsga=len(nb1), hv_ratio=hv(nb1, nb2)/hv(fb1, fb2),
        ov_model=ov(pick_model), ov_model_exact=ov(pick_exact), ov_form=ov(pick_form),
        ov_prev=len(prevxi & actual), ov_freq10=len(freq & set(actual)), ov_random=rand_ov,
        actual_dominated=dominated,
        actual_feasible=bool(keeper[act_ix].sum() >= 1 and bowler[act_ix].sum() >= 5)))
    detail.append((mr.date.date(), mr.event, [names[pool[i]] for i in sorted(pick_model)], [names[p] for p in sorted(actual)]))

R = pd.DataFrame(rows); R.to_csv('selection_results.csv', index=False)
ok = R[R['skip'].isna()] if 'skip' in R else R
print('matches evaluated:', len(ok), '| skipped:', len(R) - len(ok))

def ci(x):
    b = [np.mean(rng.choice(x, len(x))) for _ in range(2000)]
    return np.percentile(b, [2.5, 97.5])
print('\nReal XI se mean overlap (out of 11), 95% bootstrap CI')
for c, l in [('ov_model','NSGA-II knee XI (ML scores)'), ('ov_model_exact','Exact-front knee XI (ML scores)'),
             ('ov_form','Knee XI (rolling-form, bina ML)'), ('ov_prev','Pichle match ki XI (persistence)'),
             ('ov_freq10','Last 10 me sabse zyada khele top-11'), ('ov_random','Random valid XI')]:
    x = ok[c].values.astype(float); lo, hi = ci(x)
    print(f'{l:38s} {x.mean():5.2f}  [{lo:.2f}, {hi:.2f}]')
print('\npool size avg %.1f | NSGA-II hypervolume vs exact: mean %.4f, min %.4f' % (ok.pool.mean(), ok.hv_ratio.mean(), ok.hv_ratio.min()))
print('actual XI hamare front se dominated: %.0f%% | actual XI constraints satisfy karti: %.0f%%' % (
    100*ok.actual_dominated.mean(), 100*ok.actual_feasible.mean()))

# ---- sabse last India match: model ki XI vs real XI ----
dt, ev, pick, act = detail[-1]
print('\nLast match:', dt, ev)
print('MODEL XI :', ', '.join(pick))
print('REAL XI  :', ', '.join(act))

sanity | bat MAE base=13.772 model=12.879 | bowl MAE base=0.844 model=0.816
matches evaluated: 49 | skipped: 0

Real XI se mean overlap (out of 11), 95% bootstrap CI
NSGA-II knee XI (ML scores)             8.80  [8.49, 9.12]
Exact-front knee XI (ML scores)         8.78  [8.45, 9.10]
Knee XI (rolling-form, bina ML)         8.92  [8.63, 9.24]
Pichle match ki XI (persistence)        9.24  [8.88, 9.59]
Last 10 me sabse zyada khele top-11     8.20  [7.84, 8.61]
Random valid XI                         8.50  [8.26, 8.76]

pool size avg 14.4 | NSGA-II hypervolume vs exact: mean 0.9999, min 0.9958
actual XI hamare front se dominated: 65% | actual XI constraints satisfy karti: 92%

Last match: 2026-10-06 West Indies tour of India
MODEL XI : Arshdeep Singh, AR Patel, Ishan Kishan, SS Iyer, Kuldeep Yadav, SV Samson, S Dube, Tilak Varma, MP Yadav, Abhishek Sharma, Naman Dhir
REAL XI  : Arshdeep Singh, AR Patel, Ishan Kishan, SS Iyer, Kuldeep Yadav, SV Samson, S Dube, Tilak Varma, MP Yadav, Abhishek

In [4]:
from IPython.display import display, Markdown

# ---------- 1) Main results table ----------
methods = [
    ('ov_model',       'NSGA-II knee XI (ML scores)'),
    ('ov_model_exact', 'Exact-front knee XI (ML scores)'),
    ('ov_form',        'Knee XI (rolling form, no ML)'),
    ('ov_prev',        'Previous match XI (persistence)'),
    ('ov_freq10',      'Top-11 by appearances in last 10'),
    ('ov_random',      'Random valid XI from squad pool'),
]
rows = []
for col, label in methods:
    x = ok[col].values.astype(float)
    lo, hi = ci(x)
    rows.append({'Method': label, 'Mean overlap (out of 11)': x.mean(),
                 '95% CI low': lo, '95% CI high': hi})
S = pd.DataFrame(rows).set_index('Method')

display(Markdown(f'## Playing XI selection backtest: India T20Is 2025+ ({len(ok)} matches)'))
display(Markdown('**Overlap of each method\'s XI with the real India XI (higher is better)**'))
display(S.style.format('{:.2f}')
          .background_gradient(cmap='Blues', subset=['Mean overlap (out of 11)']))

# ---------- 2) Validation of NSGA-II and model summary ----------
V = pd.DataFrame({
    'Metric': ['Matches evaluated',
               'Average squad pool size',
               'NSGA-II hypervolume vs exact front (mean)',
               'NSGA-II hypervolume vs exact front (min)',
               'Real XI dominated by our Pareto front',
               'Real XI satisfies our constraints'],
    'Value': [f'{len(ok)}',
              f'{ok.pool.mean():.1f}',
              f'{ok.hv_ratio.mean():.4f}',
              f'{ok.hv_ratio.min():.4f}',
              f'{100*ok.actual_dominated.mean():.0f}%',
              f'{100*ok.actual_feasible.mean():.0f}%']
}).set_index('Metric')
display(Markdown('**Optimizer validation and diagnostics**'))
display(V)

# ---------- 3) Last match: model XI vs real XI ----------
FULL = {  # optional: abbreviated Cricsheet names -> full names (add more if you like)
    'V Suryavanshi': 'Vaibhav Sooryavanshi', 'SS Iyer': 'Shreyas Iyer', 'S Dube': 'Shivam Dube',
    'RK Singh': 'Rinku Singh', 'Ishan Kishan': 'Ishan Kishan', 'Tilak Varma': 'Tilak Varma',
    'Ravi Bishnoi': 'Ravi Bishnoi', 'Abhishek Sharma': 'Abhishek Sharma', 'Yash Thakur': 'Yash Thakur',
    'Suryansh Shedge': 'Suryansh Shedge',
}
dt, ev, pick, act = detail[-1]
allp = sorted(set(pick) | set(act), key=lambda p: (p not in pick or p not in act, p))
T = pd.DataFrame({
    'Player': [FULL.get(p, p) for p in allp],
    'Model XI': ['Yes' if p in pick else '-' for p in allp],
    'Real XI':  ['Yes' if p in act  else '-' for p in allp],
    'Result':   ['Match' if (p in pick and p in act) else ('Model only' if p in pick else 'Real only') for p in allp],
}).reset_index(drop=True)
T.index = T.index + 1

def colour(v):
    return {'Match': 'background-color:#1f5f3a', 'Model only': 'background-color:#7a5c12',
            'Real only': 'background-color:#7a2a2a'}.get(v, '')
display(Markdown(f'## Latest match: {dt}, {ev}'))
display(Markdown(f'**Overlap: {sum(p in act for p in pick)}/11 players**'))
display(T.style.map(colour, subset=['Result']))

## Playing XI selection backtest: India T20Is 2025+ (49 matches)

**Overlap of each method's XI with the real India XI (higher is better)**

,Mean overlap (out of 11),95% CI low,95% CI high
Method,,,
NSGA-II knee XI (ML scores),8.80,8.47,9.12
Exact-front knee XI (ML scores),8.78,8.45,9.12
"Knee XI (rolling form, no ML)",8.92,8.61,9.22
Previous match XI (persistence),9.24,8.90,9.59
Top-11 by appearances in last 10,8.20,7.82,8.59
Random valid XI from squad pool,8.50,8.26,8.76


**Optimizer validation and diagnostics**

,Value
Metric,
Matches evaluated,49
Average squad pool size,14.4
NSGA-II hypervolume vs exact front (mean),0.9999
NSGA-II hypervolume vs exact front (min),0.9958
Real XI dominated by our Pareto front,65%
Real XI satisfies our constraints,92%


## Latest match: 2026-10-06, West Indies tour of India

**Overlap: 11/11 players**

,Player,Model XI,Real XI,Result
1,AR Patel,Yes,Yes,Match
2,Abhishek Sharma,Yes,Yes,Match
3,Arshdeep Singh,Yes,Yes,Match
4,Ishan Kishan,Yes,Yes,Match
5,Kuldeep Yadav,Yes,Yes,Match
6,MP Yadav,Yes,Yes,Match
7,Naman Dhir,Yes,Yes,Match
8,Shivam Dube,Yes,Yes,Match
9,Shreyas Iyer,Yes,Yes,Match
10,SV Samson,Yes,Yes,Match


In [13]:
# ===== Cell 7 (honest version): NZ vs IND, 1st T20I, 22 Oct 2026, Christchurch =====
# Cell 5 ke BAAD isi kernel me chalao. Reuse: mb, mw, bat_f, bowl_f, pred, fs, asof_row,
# pareto_exact, knee, names, real, ind, matches, xi
#
# Isme 3 cheezein "honest" hain:
#  1) POOL_MODE: 'squad' (announced squad) ya 'past45' (backtest jaisa proxy, lekin SIRF PEECHE dekhta hai, future leak nahi)
#  2) Har player ka last India T20I kab tha (gap) dikhata hai, aur "returning" (365+ din gap) players flag karta hai
#  3) Sensitivity: XI ek baar poore pool ke saath, ek baar returning players hata ke. Farak = un players ka asar
import numpy as np, pandas as pd

MATCH_DATE = pd.Timestamp('2026-10-22')
OPP, VENUE_KEY = 'New Zealand', 'Hagley'
POOL_MODE = 'squad'            # 'squad' ya 'past45'
PAST_DAYS = 45
RETURN_GAP = 365               # itne din se India T20I nahi khela => "returning"
OUT_CSV = 'pred_nz_t20i_1_honest.csv'

SQUAD_KEYS = {
    'Shreyas Iyer': 'iyer', 'Tilak Varma': 'tilak', 'Abhishek Sharma': 'abhishek',
    'Vaibhav Sooryavanshi': 'suryavanshi|sooryavanshi', 'Ishan Kishan': 'kishan',
    'Sanju Samson': 'samson', 'Shivam Dube': 'dube', 'Kuldeep Yadav': 'kuldeep',
    'Axar Patel': 'patel', 'Nitish Kumar Reddy': 'reddy', 'Arshdeep Singh': 'arshdeep',
    'Prince Yadav': 'prince', 'Mayank Yadav': 'mp yadav|mayank', 'Naman Dhir': 'dhir',
    'Bhuvneshwar Kumar': 'bhuvneshwar', 'Varun Chakaravarthy': 'varun',
}
OVERRIDE = {'Axar Patel': 'AR Patel', 'Sanju Samson': 'SV Samson', 'Mayank Yadav': 'MP Yadav',
            'Bhuvneshwar Kumar': 'B Kumar', 'Shreyas Iyer': 'SS Iyer'}

# ---------- 1) pool ----------
if POOL_MODE == 'squad':
    cnt = ind[ind.date >= '2021-01-01'].groupby(['player_id', 'player']).size().rename('n').reset_index()
    pool, bad = [], []
    for label, pat in SQUAD_KEYS.items():
        if label in OVERRIDE:
            c = cnt[cnt.player == OVERRIDE[label]]
            if len(c) >= 1: pool.append(c.sort_values('n').player_id.iloc[-1]); continue
        else:
            c = cnt[cnt.player.str.lower().str.contains(pat)]
            if len(c) == 1: pool.append(c.player_id.iloc[0]); continue
        bad.append((label, c.player.tolist()))
    if bad:
        for b in bad: print('RESOLVE:', b)
        raise ValueError('OVERRIDE dict me exact Cricsheet naam daalo')
elif POOL_MODE == 'past45':
    w = ind[(ind.date < MATCH_DATE) & (ind.date >= MATCH_DATE - pd.Timedelta(days=PAST_DAYS))]
    pool = sorted(w.player_id.unique())
else:
    raise ValueError("POOL_MODE 'squad' ya 'past45' hona chahiye")
if not (11 <= len(pool) <= 22): raise ValueError(f'pool size {len(pool)}: enumeration ke liye 11-22 chahiye')
print(f'POOL_MODE={POOL_MODE} | pool ({len(pool)}):', [names[p] for p in pool])

# last India T20I (data me) aur gap
last_intl = ind.groupby('player_id').date.max()
gap = {p: (MATCH_DATE - last_intl[p]).days if p in last_intl.index else 9999 for p in pool}
returning = [p for p in pool if gap[p] > RETURN_GAP]
print('Returning players (India T20I gap >%d din):' % RETURN_GAP,
      [(names[p], gap[p]) for p in returning] or 'koi nahi')

# ---------- 2) context features ----------
ti = real.drop_duplicates(['match_id', 'team']).dropna(subset=['team_total']).sort_values('date')
gl = ti[ti.date < pd.Timestamp('2024-01-01')].team_total.mean()
last20 = lambda team, col: ti[(ti.team == team) & (ti.date < MATCH_DATE)][col].tail(20).mean()
v = ti[ti.venue.fillna('').str.contains(VENUE_KEY)]
print('venue strings:', v.venue.unique(), '| innings at venue:', len(v))
ctx = dict(
    venue_avg_total    = (v.team_total.sum() + 5 * gl) / (len(v) + 5),
    team_bat_form      = last20('India', 'team_total'),
    team_conceded_form = last20('India', 'opp_total'),
    opp_conceded_form  = last20(OPP, 'opp_total'),
    opp_bat_form       = last20(OPP, 'team_total'),
)
print('context:', {k: round(float(x), 1) for k, x in ctx.items()})

# ---------- 3) selection function (kisi bhi pool ke liye) ----------
def solve(pool_ids):
    pos = [asof_row(p, MATCH_DATE) for p in pool_ids]
    X = fs.loc[pos].copy().reset_index(drop=True)
    X['days_since'] = (MATCH_DATE - X.prev_date).dt.days.fillna(365).clip(upper=365)
    X['is_intl'] = 1
    for k, val in ctx.items(): X[k] = val
    keeper = (X.stump_prev.values >= 2).astype(float)
    bowler = (X.p_bowl_15.values >= 0.35).astype(float)
    er, ew = pred(mb, X[bat_f]), pred(mw, X[bowl_f])
    bat, bowl = X.p_bat_15.values * er, X.p_bowl_15.values * ew
    ex = pareto_exact(bat, bowl, keeper, bowler)
    if ex is None: return None
    C, FC, fb1, fb2, _, _ = ex
    return dict(pool=pool_ids, X=X, er=er, ew=ew, bat=bat, bowl=bowl, keeper=keeper, bowler=bowler,
                nC=len(C), FC=FC, fb1=fb1, fb2=fb2, kk=knee(fb1, fb2),
                kb=int(np.argmax(fb2)), kt=int(np.argmax(fb1)))

def players_of(R, k): return [names[R['pool'][i]] for i in sorted(R['FC'][k])]

def report(R, title):
    print(f'\n=== {title} ===')
    print(f"feasible XIs: {R['nC']} | Pareto-optimal: {len(R['FC'])}")
    for lab, k in [('Bowling-extreme', R['kb']), ('KNEE (pick)', R['kk']), ('Batting-extreme', R['kt'])]:
        print(f"{lab:16s} f1={R['fb1'][k]:7.2f} f2={R['fb2'][k]:5.2f} | " + ', '.join(players_of(R, k)))
    freq = np.bincount(R['FC'].ravel(), minlength=len(R['pool'])) / len(R['FC'])
    print('Player frequency on front:')
    for i in np.argsort(-freq): print(f"  {names[R['pool'][i]]:22s} {freq[i]:.0%}")

# ---------- 4) FULL pool ----------
R1 = solve(pool)
if R1 is None: raise ValueError('infeasible: keeper (stump_prev>=2) ya 5 bowlers (p_bowl_15>=0.35) nahi')
X1 = R1['X']
tab = pd.DataFrame({'player': [names[p] for p in pool], 'matches_hist': X1.matches_prev.values,
                    'days_since_any': X1.days_since.values, 'gap_intl_days': [gap[p] for p in pool],
                    'p_bat': X1.p_bat_15.round(2), 'E_runs|bat': R1['er'].round(1),
                    'p_bowl': X1.p_bowl_15.round(2), 'E_wkts|bowl': R1['ew'].round(2),
                    'bat_i': R1['bat'].round(1), 'bowl_i': R1['bowl'].round(2)})
print(tab.to_string(index=False))
report(R1, 'FULL POOL')

# ---------- 5) SENSITIVITY: returning players hata ke ----------
R2 = None
if returning:
    pool2 = [p for p in pool if p not in returning]
    if len(pool2) >= 11:
        R2 = solve(pool2)
    if R2 is None:
        print('\nReturning players hatane par pool/constraints infeasible')
    else:
        report(R2, 'WITHOUT returning players: ' + ', '.join(names[p] for p in returning))
        k1 = {pool[i] for i in R1['FC'][R1['kk']]}
        k2 = {R2['pool'][i] for i in R2['FC'][R2['kk']]}
        print(f'\nKnee XI overlap (full vs without returning): {len(k1 & k2)} / 11')
        print('Sirf full me   :', sorted(names[p] for p in k1 - k2))
        print('Sirf without me:', sorted(names[p] for p in k2 - k1))

# ---------- 6) honest baseline + save ----------
prev = matches.index[-1]; prevxi = xi[prev]
k1set = {pool[i] for i in R1['FC'][R1['kk']]}
print(f'\nPichla India match in data: {matches.loc[prev, "date"].date()} ({matches.loc[prev, "event"]})')
print('Prev XI se knee XI ka overlap:', len(k1set & prevxi), '/ 11')

rows = []
for tag, R in [('full', R1), ('no_returning', R2)]:
    if R is None: continue
    for lab, k in [('bowling_extreme', R['kb']), ('knee', R['kk']), ('batting_extreme', R['kt'])]:
        rows.append(dict(pool=tag, kind=lab, f1=R['fb1'][k], f2=R['fb2'][k], players='; '.join(players_of(R, k))))
rows.append(dict(pool='-', kind='prev_match_xi', f1=np.nan, f2=np.nan, players='; '.join(names[p] for p in sorted(prevxi))))
rows.append(dict(pool='-', kind='data_last_match', f1=np.nan, f2=np.nan, players=str(matches.loc[prev, 'date'].date())))
rows.append(dict(pool='-', kind='meta', f1=np.nan, f2=np.nan, players=f'POOL_MODE={POOL_MODE}; pool={len(pool)}'))
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print('saved ->', OUT_CSV)

# ---------- 7) MATCH KE BAAD ----------
# REAL_XI = ['SS Iyer', ...]   # pool print me jo naam dikhe wahi
# real_set = {p for p in pool if names[p] in REAL_XI}
# print('Knee XI overlap :', len(k1set & real_set), '/ 11')
# print('Prev XI overlap :', len(prevxi & real_set), '/ 11')

POOL_MODE=squad | pool (16): ['SS Iyer', 'Tilak Varma', 'Abhishek Sharma', 'V Suryavanshi', 'Ishan Kishan', 'SV Samson', 'S Dube', 'Kuldeep Yadav', 'AR Patel', 'Nithish Kumar Reddy', 'Arshdeep Singh', 'Prince Yadav', 'MP Yadav', 'Naman Dhir', 'B Kumar', 'CV Varun']
Returning players (India T20I gap >365 din): [('Nithish Kumar Reddy', 638), ('B Kumar', 1430)]
venue strings: ['Hagley Oval' 'Hagley Oval, Christchurch'] | innings at venue: 29
context: {'venue_avg_total': 152.1, 'team_bat_form': 175.2, 'team_conceded_form': 165.3, 'opp_conceded_form': 168.3, 'opp_bat_form': 165.2}
             player  matches_hist  days_since_any  gap_intl_days  p_bat  E_runs|bat  p_bowl  E_wkts|bowl  bat_i  bowl_i
            SS Iyer           211              16             16   1.00   29.299999    0.00         0.54   29.3    0.00
        Tilak Varma           129              16             16   0.93   21.700001    0.07         0.45   20.2    0.03
    Abhishek Sharma           152              16        

In [7]:
# ===== Cell 7: NZ vs IND, 1st T20I, 22 Oct 2026, Christchurch (Hagley Oval) =====
# Cell 5 ke BAAD isi kernel me chalao. Ye cell 5 ke objects reuse karta hai:
# mb, mw, bat_f, bowl_f, pred, fs, asof_row, pareto_exact, knee, nsga2, names, real, ind, matches, xi
import numpy as np, pandas as pd

MATCH_DATE = pd.Timestamp('2026-10-22')
OPP, VENUE_KEY = 'New Zealand', 'Hagley'            # Cricsheet venue string me ye hona chahiye
OUT_CSV = 'pred_nz_t20i_1.csv'                       # prediction save hogi, match ke baad compare karne ke liye

# ---------- 1) squad -> Cricsheet player_id ----------
# squad: Sports Digest (6 Oct 2026), 16 players. Match se pehle confirm kar lena.
SQUAD_KEYS = {
    'Shreyas Iyer': 'iyer', 'Tilak Varma': 'tilak', 'Abhishek Sharma': 'abhishek',
    'Vaibhav Sooryavanshi': 'suryavanshi|sooryavanshi', 'Ishan Kishan': 'kishan',
    'Sanju Samson': 'samson', 'Shivam Dube': 'dube', 'Kuldeep Yadav': 'kuldeep',
    'Axar Patel': 'patel', 'Nitish Kumar Reddy': 'reddy', 'Arshdeep Singh': 'arshdeep',
    'Prince Yadav': 'prince', 'Mayank Yadav': 'mp yadav|mayank', 'Naman Dhir': 'dhir',
    'Bhuvneshwar Kumar': 'bhuvneshwar', 'Varun Chakaravarthy': 'varun',
}
# Cricsheet me naam abbreviated hote hain, isliye ambiguous/abbreviated naam yahan pin karo.
# AR Patel, SV Samson, MP Yadav, SS Iyer: pichle outputs se confirmed. B Kumar: Bhuvneshwar ke liye andaza, pool print me verify karna.
OVERRIDE = {'Axar Patel': 'AR Patel', 'Sanju Samson': 'SV Samson', 'Mayank Yadav': 'MP Yadav',
            'Bhuvneshwar Kumar': 'B Kumar', 'Shreyas Iyer': 'SS Iyer'}

# Bhuvneshwar ka last India T20I Nov 2022 ka hai, isliye window 2021 se rakhi hai (2024 se nahi)
cnt = ind[ind.date >= '2021-01-01'].groupby(['player_id', 'player']).size().rename('n').reset_index()
pool, bad = [], []
for label, pat in SQUAD_KEYS.items():
    if label in OVERRIDE:
        c = cnt[cnt.player == OVERRIDE[label]]
    else:
        c = cnt[cnt.player.str.lower().str.contains(pat)]
    if len(c) >= 1 and label in OVERRIDE:
        pool.append(c.sort_values('n').player_id.iloc[-1])       # same abbreviation ke kayi log ho to sabse zyada khelne wala
    elif len(c) == 1:
        pool.append(c.player_id.iloc[0])
    else:
        bad.append((label, c.player.tolist()))
if bad:
    for b in bad: print('RESOLVE:', b)
    raise ValueError('OVERRIDE dict me exact Cricsheet naam daalo aur cell dobara chalao')
print('pool (%d):' % len(pool), [names[p] for p in pool])

# ---------- 2) context features (future match ki row nahi hai, to khud banao) ----------
ti = real.drop_duplicates(['match_id', 'team']).dropna(subset=['team_total']).sort_values('date')
gl = ti[ti.date < pd.Timestamp('2024-01-01')].team_total.mean()
last20 = lambda team, col: ti[(ti.team == team) & (ti.date < MATCH_DATE)][col].tail(20).mean()
v = ti[ti.venue.fillna('').str.contains(VENUE_KEY)]
print('venue strings:', v.venue.unique(), '| innings at venue:', len(v))
if len(v) == 0: print('WARNING: venue match nahi hua, venue_avg_total = global mean par fallback')
ctx = dict(
    venue_avg_total    = (v.team_total.sum() + 5 * gl) / (len(v) + 5),
    team_bat_form      = last20('India', 'team_total'),
    team_conceded_form = last20('India', 'opp_total'),
    opp_conceded_form  = last20(OPP, 'opp_total'),
    opp_bat_form       = last20(OPP, 'team_total'),
)
print('context:', {k: round(float(x), 1) for k, x in ctx.items()})

# ---------- 3) as-of features, predictions, objectives ----------
pos = [asof_row(p, MATCH_DATE) for p in pool]
X = fs.loc[pos].copy().reset_index(drop=True)
X['days_since'] = (MATCH_DATE - X.prev_date).dt.days.fillna(365).clip(upper=365)
X['is_intl'] = 1
for k, val in ctx.items(): X[k] = val
keeper = (X.stump_prev.values >= 2).astype(float)
bowler = (X.p_bowl_15.values >= 0.35).astype(float)
er, ew = pred(mb, X[bat_f]), pred(mw, X[bowl_f])
bat, bowl = X.p_bat_15.values * er, X.p_bowl_15.values * ew
pn = [names[p] for p in pool]

tab = pd.DataFrame({'player': pn, 'matches_hist': X.matches_prev.values, 'days_since': X.days_since.values,
                    'p_bat': X.p_bat_15.round(2), 'E_runs|bat': er.round(1),
                    'p_bowl': X.p_bowl_15.round(2), 'E_wkts|bowl': ew.round(2),
                    'bat_i': bat.round(1), 'bowl_i': bowl.round(2), 'keeper': keeper, 'bowler': bowler})
print(tab.to_string(index=False))   # p_bat/p_bowl ~0 wale players selection me aa hi nahi payenge

# ---------- 4) exact front + NSGA-II + knee ----------
ex = pareto_exact(bat, bowl, keeper, bowler)
if ex is None: raise ValueError('infeasible: squad me keeper (stump_prev>=2) ya 5 bowlers (p_bowl_15>=0.35) nahi')
C, FC, fb1, fb2, _, _ = ex
ns = nsga2(bat, bowl, keeper, bowler)
print(f'\nfeasible XIs: {len(C)} | Pareto-optimal: {len(FC)}')
if ns is not None:
    Xn, nb1, nb2 = ns
    print('NSGA-II front size:', len(nb1))

def show(label, k):
    s = sorted(FC[k])
    print(f'{label:16s} f1={fb1[k]:7.2f} f2={fb2[k]:5.2f} | ' + ', '.join(pn[i] for i in s))
kb = int(np.argmax(fb2)); kk = knee(fb1, fb2); kt = int(np.argmax(fb1))
show('Bowling-extreme', kb)
show('KNEE (pick)',     kk)
show('Batting-extreme', kt)

freq = np.bincount(FC.ravel(), minlength=len(pool)) / len(FC)
print('\nPlayer frequency on front:')
for i in np.argsort(-freq): print(f'  {pn[i]:22s} {freq[i]:.0%}')

# ---------- 5) honest baseline: pichle India match ka XI ----------
prev = matches.index[-1]
prevxi = xi[prev]
kneeset = {pool[i] for i in FC[kk]}
print(f'\nPichla India match in data: {matches.loc[prev, "date"].date()} ({matches.loc[prev, "event"]})')
print('Prev XI se knee XI ka overlap:', len(kneeset & prevxi), '/ 11')
print('Prev XI:', ', '.join(names[p] for p in sorted(prevxi)))

# ---------- 6) prediction save karo (match se pehle), baad me real XI se compare ----------
rows = []
for label, k in [('bowling_extreme', kb), ('knee', kk), ('batting_extreme', kt)]:
    rows.append(dict(kind=label, f1=fb1[k], f2=fb2[k], players='; '.join(pn[i] for i in sorted(FC[k]))))
rows.append(dict(kind='prev_match_xi', f1=np.nan, f2=np.nan, players='; '.join(names[p] for p in sorted(prevxi))))
rows.append(dict(kind='data_last_match', f1=np.nan, f2=np.nan, players=str(matches.loc[prev, 'date'].date())))
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print('saved ->', OUT_CSV)

# ---------- 7) MATCH KE BAAD: real XI daal ke ye block chalao ----------
# REAL_XI = ['A Name', 'B Name', ...]   # Cricsheet jaise naam (pool print me jo naam dikhe wahi)
# real_set = {p for p in pool if names[p] in REAL_XI}
# print('Knee XI overlap :', len(kneeset & real_set), '/ 11')
# print('Prev XI overlap :', len(prevxi & real_set), '/ 11')
# print('Real XI (pool me):', sorted(names[p] for p in real_set))

pool (16): ['SS Iyer', 'Tilak Varma', 'Abhishek Sharma', 'V Suryavanshi', 'Ishan Kishan', 'SV Samson', 'S Dube', 'Kuldeep Yadav', 'AR Patel', 'Nithish Kumar Reddy', 'Arshdeep Singh', 'Prince Yadav', 'MP Yadav', 'Naman Dhir', 'B Kumar', 'CV Varun']
venue strings: ['Hagley Oval' 'Hagley Oval, Christchurch'] | innings at venue: 29
context: {'venue_avg_total': 152.1, 'team_bat_form': 175.2, 'team_conceded_form': 165.3, 'opp_conceded_form': 168.3, 'opp_bat_form': 165.2}
             player  matches_hist  days_since  p_bat  E_runs|bat  p_bowl  E_wkts|bowl  bat_i  bowl_i  keeper  bowler
            SS Iyer           211          16   1.00   29.299999    0.00         0.54   29.3    0.00     0.0     0.0
        Tilak Varma           129          16   0.93   21.700001    0.07         0.45   20.2    0.03     0.0     0.0
    Abhishek Sharma           152          16   1.00   27.600000    0.27         0.45   27.6    0.12     0.0     0.0
      V Suryavanshi            31          19   1.00   32.0999